In [2]:
# ==============================================================
# Hindi Morphological Tagger — Multi-Dataset + Semi-Automatic
# Datasets: UD Hindi HDTB 
# ==============================================================

import os, sys, subprocess, socket, warnings, json, random, io, zipfile
import urllib.request
from pathlib import Path
from collections import defaultdict, Counter

# ---------- 0a. NETWORK RESILIENCE ----------
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"]   = "900"
os.environ["HF_HUB_ETAG_TIMEOUT"]       = "60"
os.environ["HF_HUB_DISABLE_TELEMETRY"]  = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# ---------- 0b. INSTALL DEPENDENCIES ----------
def _pip(*pkgs):
    for p in pkgs:
        try:
            __import__(p.split("[")[0].replace("-", "_"))
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])

_pip("numpy", "pandas", "scikit-learn", "statsmodels", "openpyxl")
_pip("torch", "datasets", "transformers", "seqeval")
_pip("huggingface_hub[hf_transfer]")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from seqeval.metrics import classification_report as seq_report
from statsmodels.stats.inter_rater import fleiss_kappa

warnings.filterwarnings("ignore")
print("✅ Dependencies ready.")

# ==============================================================
# 1. PATHS
# ==============================================================
OUTPUT_DIR   = Path(r"D:\DATASET On 10-12-2025\Objective 1\OB1.1\Dataset\Morph_Tagger_Report")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
UD_CACHE     = OUTPUT_DIR / "ud_hindi_hdtb"
LDC_CACHE    = OUTPUT_DIR / "ldc2010t24"
INDIC_CACHE  = OUTPUT_DIR / "indiccorpus"

# ==============================================================
# 2. LOAD UD HINDI HDTB (primary supervised corpus)
# ==============================================================
print("\n[1/7] Loading UD Hindi HDTB ...")

def _download_with_retries(url, dest, max_retries=5):
    for attempt in range(1, max_retries + 1):
        try:
            print(f"   Attempt {attempt}/{max_retries} ...")
            with urllib.request.urlopen(url, timeout=120) as r, open(dest, "wb") as f:
                while True:
                    chunk = r.read(1 << 20)
                    if not chunk: break
                    f.write(chunk)
            return True
        except Exception as e:
            print(f"   ⚠ {e}")
    return False

def parse_conllu(path):
    sentences, sent = [], []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line or line.startswith("#"):
                if sent: sentences.append(sent); sent = []
                continue
            parts = line.split("\t")
            if len(parts) >= 10 and parts[0].isdigit():
                feats = {}
                if parts[5] != "_":
                    for feat in parts[5].split("|"):
                        if "=" in feat:
                            k, v = feat.split("=", 1)
                            feats[k] = v
                sent.append({"form": parts[1], "upos": parts[3], "feats": feats})
    if sent: sentences.append(sent)
    return sentences

def normalize_split(sentences):
    return [{"tokens": [w["form"] for w in s],
             "upos":   [w["upos"] for w in s],
             "feats":  [w["feats"] for w in s]} for s in sentences]

ud_train = ud_dev = ud_test = None
try:
    from datasets import load_dataset
    ud = load_dataset("universal_dependencies", "hi_hdtb")
    ud_train = [{"tokens": ex["tokens"], "upos": ex["upos"], "feats": ex["feats"]} for ex in ud["train"]]
    ud_dev   = [{"tokens": ex["tokens"], "upos": ex["upos"], "feats": ex["feats"]} for ex in ud["validation"]]
    ud_test  = [{"tokens": ex["tokens"], "upos": ex["upos"], "feats": ex["feats"]} for ex in ud["test"]]
    print(f"   ✓ UD HDTB: train={len(ud_train)}, dev={len(ud_dev)}, test={len(ud_test)}")
except Exception:
    print("   ⚠ datasets library failed; using GitHub zip ...")
    UD_CACHE.mkdir(parents=True, exist_ok=True)
    zip_path = UD_CACHE / "ud_hindi_hdtb.zip"
    base_ok = UD_CACHE / "UD_Hindi-HDTB-master" / "hi_hdtb-ud-train.conllu"
    if not base_ok.exists():
        if not zip_path.exists():
            ok = _download_with_retries(
                "https://github.com/UniversalDependencies/UD_Hindi-HDTB/archive/refs/heads/master.zip", zip_path)
            if not ok: raise RuntimeError("Could not download UD Hindi HDTB.")
        with zipfile.ZipFile(zip_path) as z: z.extractall(UD_CACHE)
    base = UD_CACHE / "UD_Hindi-HDTB-master"
    ud_train = normalize_split(parse_conllu(base / "hi_hdtb-ud-train.conllu"))
    ud_dev   = normalize_split(parse_conllu(base / "hi_hdtb-ud-dev.conllu"))
    ud_test  = normalize_split(parse_conllu(base / "hi_hdtb-ud-test.conllu"))
    print(f"   ✓ UD HDTB: train={len(ud_train)}, dev={len(ud_dev)}, test={len(ud_test)}")

# ==============================================================
# 3. LOAD LDC2010T24 (additional supervised corpus)
# ==============================================================
print("\n[2/7] Loading LDC2010T24 ...")

def load_ldc2010t24(ldc_dir):
    """
    Load LDC2010T24 from its XML or text files.
    Expected format: XML with <Sentence> and <Word> elements,
    each Word has attributes for category, gender, number, case.
    """
    sentences = []
    xml_files = list(ldc_dir.rglob("*.xml")) + list(ldc_dir.rglob("*.txt"))
    if not xml_files:
        return None

    import xml.etree.ElementTree as ET
    for xml_path in xml_files:
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()
            for sent_elem in root.iter("Sentence"):
                tokens, upos, feats = [], [], []
                for word in sent_elem.iter("Word"):
                    form = word.get("form") or word.text or ""
                    cat  = word.get("category", "X")
                    # LDC2010T24 uses attributes like gender, number, case
                    f = {}
                    if word.get("gender"): f["Gender"] = word.get("gender")
                    if word.get("number"): f["Number"] = word.get("number")
                    if word.get("case"):   f["Case"]   = word.get("case")
                    tokens.append(form.strip())
                    upos.append(cat)
                    feats.append(f)
                if tokens:
                    sentences.append({"tokens": tokens, "upos": upos, "feats": feats})
        except Exception:
            continue
    return sentences if sentences else None

ldc_data = None
if LDC_CACHE.exists():
    ldc_data = load_ldc2010t24(LDC_CACHE)
    if ldc_data:
        print(f"   ✓ LDC2010T24 loaded: {len(ldc_data)} sentences")
    else:
        print("   ⚠ LDC2010T24 folder found but no parseable XML/text files.")
else:
    print("   ⚠ LDC2010T24 not found on disk.")
    print("   → LDC2010T24 requires a license from the Linguistic Data Consortium.")
    print("   → If you have the files, place them in:")
    print(f"      {LDC_CACHE}")
    print("   → Continuing with UD HDTB only for supervised training.")

# Merge LDC data into training set if available
if ldc_data:
    # Split LDC into 90/10 train/dev
    random.seed(42)
    random.shuffle(ldc_data)
    split_idx = int(0.9 * len(ldc_data))
    ud_train.extend(ldc_data[:split_idx])
    ud_dev.extend(ldc_data[split_idx:])
    print(f"   ✓ Merged LDC into training. New total: train={len(ud_train)}, dev={len(ud_dev)}")

# ==============================================================
# 4. LOAD AI4BHARAT INDICCORPUS (for semi-automatic annotation)
# ==============================================================
print("\n[3/7] Loading AI4Bharat IndicCorpus (for semi-automatic annotation) ...")
print("   This corpus is unannotated. It will be used to pre-annotate")
print("   Hindi sentences with the trained tagger, producing a larger")
print("   pool that can be manually corrected for future iterations.")

indic_sentences = []
INDIC_CACHE.mkdir(parents=True, exist_ok=True)
indic_file = INDIC_CACHE / "hi.txt"

if not indic_file.exists():
    # AI4Bharat IndicCorp Hindi download URL (63.1M sentences)
    INDIC_URL = "https://objectstore.e2enetworks.net/ai4b-public-nlu-nlg/v1-indiccorp/hi.txt"
    print(f"   Downloading IndicCorpus Hindi (this may take a while) ...")
    ok = _download_with_retries(INDIC_URL, indic_file, max_retries=3)
    if not ok:
        print("   ⚠ Could not download IndicCorpus. Skipping semi-automatic step.")
        indic_file = None

if indic_file and indic_file.exists():
    # Read first N sentences (to keep memory manageable)
    N_INDIC = 10000
    with open(indic_file, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= N_INDIC: break
            line = line.strip()
            if len(line.split()) >= 3:
                indic_sentences.append(line)
    print(f"   ✓ Loaded {len(indic_sentences)} Hindi sentences from IndicCorpus.")

# ==============================================================
# 5. BUILD LABEL SETS
# ==============================================================
print("\n[4/7] Building label sets ...")

def build_label_set(data, task):
    labels = set()
    for sent in data:
        for feat in sent["feats"]:
            if task in feat: labels.add(feat[task])
        if task == "upos": labels.update(sent["upos"])
    return sorted(labels)

label_maps = {
    "upos":   build_label_set(ud_train, "upos"),
    "Gender": build_label_set(ud_train, "Gender"),
    "Number": build_label_set(ud_train, "Number"),
    "Case":   build_label_set(ud_train, "Case"),
}
for task in ["Gender", "Number", "Case"]:
    if "O" not in label_maps[task]: label_maps[task].append("O")

label2id = {task: {l: i for i, l in enumerate(labels)} for task, labels in label_maps.items()}
id2label = {task: {i: l for l, i in m.items()} for task, m in label2id.items()}

print(f"   POS: {len(label_maps['upos'])} labels")
print(f"   Gender: {label_maps['Gender']}")
print(f"   Number: {label_maps['Number']}")
print(f"   Case: {label_maps['Case']}")

# ==============================================================
# 6. COMPUTE CLASS WEIGHTS (for minority class improvement)
# ==============================================================
print("\n[5/7] Computing class weights for imbalanced labels ...")

def compute_class_weights(data, task, id2label_map):
    counts = Counter()
    n_classes = len(id2label_map)
    for sent in data:
        if task == "upos":
            for u in sent["upos"]: counts[u] += 1
        else:
            for feat in sent["feats"]:
                counts[feat.get(task, "O")] += 1
    total = sum(counts.values())
    weights = []
    for i in range(n_classes):
        label = id2label_map[i]
        c = counts.get(label, 1)
        weights.append(total / (n_classes * max(c, 1)))
    return torch.tensor(weights, dtype=torch.float)

# ==============================================================
# 7. MULTI-TASK MODEL WITH WEIGHTED LOSS
# ==============================================================
print("\n[6/7] Defining multi-task model with class weights ...")

class MultiTaskMorphTagger(nn.Module):
    def __init__(self, model_name, label_maps, class_weights):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.config = self.encoder.config
        hidden_size = self.config.hidden_size

        self.pos_head    = nn.Linear(hidden_size, len(label_maps["upos"]))
        self.gender_head = nn.Linear(hidden_size, len(label_maps["Gender"]))
        self.number_head = nn.Linear(hidden_size, len(label_maps["Number"]))
        self.case_head   = nn.Linear(hidden_size, len(label_maps["Case"]))

        total_feat_dim = (len(label_maps["upos"]) + len(label_maps["Gender"]) +
                          len(label_maps["Number"]) + len(label_maps["Case"]))
        self.morph_proj = nn.Linear(total_feat_dim, 128)

        # Register class weights as buffers (not trainable)
        self.register_buffer("w_pos",    class_weights["upos"])
        self.register_buffer("w_gender", class_weights["Gender"])
        self.register_buffer("w_number", class_weights["Number"])
        self.register_buffer("w_case",   class_weights["Case"])

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        seq = outputs.last_hidden_state

        pos_logits    = self.pos_head(seq)
        gender_logits = self.gender_head(seq)
        number_logits = self.number_head(seq)
        case_logits   = self.case_head(seq)

        pos_probs    = torch.softmax(pos_logits, dim=-1)
        gender_probs = torch.softmax(gender_logits, dim=-1)
        number_probs = torch.softmax(number_logits, dim=-1)
        case_probs   = torch.softmax(case_logits, dim=-1)

        combined = torch.cat([pos_probs, gender_probs, number_probs, case_probs], dim=-1)
        morph_embedding = self.morph_proj(combined)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            # Use class-weighted losses
            loss_fct_w_pos    = nn.CrossEntropyLoss(ignore_index=-100, weight=self.w_pos)
            loss_fct_w_gender = nn.CrossEntropyLoss(ignore_index=-100, weight=self.w_gender)
            loss_fct_w_number = nn.CrossEntropyLoss(ignore_index=-100, weight=self.w_number)
            loss_fct_w_case   = nn.CrossEntropyLoss(ignore_index=-100, weight=self.w_case)

            pos_loss    = loss_fct_w_pos(pos_logits.view(-1, pos_logits.size(-1)),    labels["upos"].view(-1))
            gender_loss = loss_fct_w_gender(gender_logits.view(-1, gender_logits.size(-1)), labels["Gender"].view(-1))
            number_loss = loss_fct_w_number(number_logits.view(-1, number_logits.size(-1)), labels["Number"].view(-1))
            case_loss   = loss_fct_w_case(case_logits.view(-1, case_logits.size(-1)),   labels["Case"].view(-1))
            loss = pos_loss + gender_loss + number_loss + case_loss

        return {"loss": loss, "pos_logits": pos_logits, "gender_logits": gender_logits,
                "number_logits": number_logits, "case_logits": case_logits,
                "morph_embedding": morph_embedding}

# ==============================================================
# 8. PREPARE DATASET
# ==============================================================
print("\n[7/7] Preparing dataset ...")

class MorphDataset(torch.utils.data.Dataset):
    def __init__(self, data, tokenizer, label2id, max_len=128):
        self.data, self.tok, self.l2i, self.max_len = data, tokenizer, label2id, max_len
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        ex = self.data[idx]
        enc = self.tok(ex["tokens"], is_split_into_words=True, truncation=True,
                       padding="max_length", max_length=self.max_len, return_tensors="pt")
        word_ids = enc.word_ids()
        labels = {task: [] for task in ["upos", "Gender", "Number", "Case"]}
        for task in labels:
            prev = None
            for wid in word_ids:
                if wid is None or wid != prev:
                    if task == "upos":
                        labels[task].append(self.l2i["upos"][ex["upos"][wid]] if wid is not None else -100)
                    else:
                        feat_val = ex["feats"][wid].get(task, "O") if wid is not None else "O"
                        labels[task].append(self.l2i[task][feat_val] if wid is not None else -100)
                else:
                    labels[task].append(-100)
                prev = wid
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "labels": {task: torch.tensor(v) for task, v in labels.items()}}

# Compute weights
class_weights = {
    "upos":   compute_class_weights(ud_train, "upos",   id2label["upos"]),
    "Gender": compute_class_weights(ud_train, "Gender", id2label["Gender"]),
    "Number": compute_class_weights(ud_train, "Number", id2label["Number"]),
    "Case":   compute_class_weights(ud_train, "Case",   id2label["Case"]),
}

# Load model
tokenizer, model, model_used = None, None, None
for candidate in ["google/muril-base-cased", "distilbert-base-multilingual-cased"]:
    try:
        print(f"   Trying: {candidate} ...")
        tok = AutoTokenizer.from_pretrained(candidate, use_fast=True)
        mdl = MultiTaskMorphTagger(candidate, label_maps, class_weights)
        tokenizer, model, model_used = tok, mdl, candidate
        print(f"   ✓ Loaded: {candidate}")
        break
    except Exception as e:
        print(f"   ⚠ Failed: {candidate} — {e}")
if model is None: raise RuntimeError("Could not load any model.")

# ==============================================================
# 9. TRAIN
# ==============================================================
print("\nTraining multi-task model ...")

train_ds = MorphDataset(ud_train, tokenizer, label2id)
val_ds   = MorphDataset(ud_dev,   tokenizer, label2id)
test_ds  = MorphDataset(ud_test,  tokenizer, label2id)

def custom_collate(batch):
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "labels": {task: torch.stack([b["labels"][task] for b in batch])
                   for task in ["upos", "Gender", "Number", "Case"]},
    }

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "mtl_tagger"),
    learning_rate=2e-5, per_device_train_batch_size=16,
    per_device_eval_batch_size=16, num_train_epochs=3,
    weight_decay=0.01, logging_steps=100,
    report_to="none", seed=42, fp16=torch.cuda.is_available(),
    save_strategy="no", disable_tqdm=True,
    dataloader_num_workers=0, dataloader_pin_memory=False,
)

trainer = Trainer(model=model, args=training_args,
                  train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=custom_collate)
trainer.train()

# ==============================================================
# 10. EVALUATE ON UD TEST SPLIT
# ==============================================================
print("\nEvaluating on held-out UD Hindi test split ...")
model.eval()
device = next(model.parameters()).device

metrics = {task: {"y_true": [], "y_pred": []} for task in ["upos", "Gender", "Number", "Case"]}

with torch.no_grad():
    for batch in torch.utils.data.DataLoader(test_ds, batch_size=16, collate_fn=custom_collate):
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        out  = model(ids, mask)
        for task, key in [("upos","pos_logits"),("Gender","gender_logits"),
                          ("Number","number_logits"),("Case","case_logits")]:
            preds = torch.argmax(out[key], dim=-1).cpu().numpy()
            true  = batch["labels"][task].numpy()
            for p, t in zip(preds, true):
                for pi, ti in zip(p, t):
                    if ti != -100:
                        metrics[task]["y_true"].append(id2label[task][ti])
                        metrics[task]["y_pred"].append(id2label[task][pi])

reports = {}
for task in metrics:
    report = seq_report([metrics[task]["y_true"]], [metrics[task]["y_pred"]], digits=4)
    reports[task] = report
    print(f"\n=== {task} Report ===")
    print(report)

# ==============================================================
# 11. SEMI-AUTOMATIC ANNOTATION ON INDICCORPUS
# ==============================================================
if indic_sentences:
    print(f"\nSemi-automatic pre-annotation of {len(indic_sentences)} IndicCorpus sentences ...")
    pre_annotated = []
    with torch.no_grad():
        for i in range(0, len(indic_sentences), 32):
            batch_texts = indic_sentences[i:i+32]
            token_lists = [t.split() for t in batch_texts]
            enc = tokenizer(token_lists, is_split_into_words=True, truncation=True,
                            padding=True, max_length=128, return_tensors="pt").to(device)
            out = model(enc["input_ids"], enc["attention_mask"])
            for j, text in enumerate(batch_texts):
                word_ids = enc.word_ids(batch_index=j)
                tokens, upos, gender, number, case = [], [], [], [], []
                seen = set()
                for k, wid in enumerate(word_ids):
                    if wid is None or wid in seen: continue
                    seen.add(wid)
                    tokens.append(token_lists[j][wid])
                    upos.append(id2label["upos"][int(torch.argmax(out["pos_logits"][j,k]))])
                    gender.append(id2label["Gender"][int(torch.argmax(out["gender_logits"][j,k]))])
                    number.append(id2label["Number"][int(torch.argmax(out["number_logits"][j,k]))])
                    case.append(id2label["Case"][int(torch.argmax(out["case_logits"][j,k]))])
                pre_annotated.append({"sentence": text, "tokens": tokens, "upos": upos,
                                      "gender": gender, "number": number, "case": case})
    # Save to file
    pre_ann_path = OUTPUT_DIR / "indiccorpus_pre_annotated.jsonl"
    with open(pre_ann_path, "w", encoding="utf-8") as f:
        for rec in pre_annotated:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"   ✓ Saved {len(pre_annotated)} pre-annotated sentences to {pre_ann_path}")
    print("   → Manually correct a subset of these and add them to future training rounds.")

# ==============================================================
# 12. FLEISS' KAPPA (replace dummy with real ratings)
# ==============================================================
print("\nComputing Fleiss' Kappa ...")
# REPLACE this dummy with your real three-annotator ratings
dummy_ratings = [[1,1,2],[2,2,2],[3,3,3],[1,2,2],[2,3,3]] * 100
categories = [1, 2, 3]
mat = np.zeros((len(dummy_ratings), len(categories)))
for i, row in enumerate(dummy_ratings):
    for j, c in enumerate(categories):
        mat[i, j] = row.count(c)
kappa = fleiss_kappa(mat, method="fleiss")
print(f"   ✓ Fleiss' Kappa (placeholder): {kappa:.4f}")

# ==============================================================
# 13. WRITE REPORT
# ==============================================================
report_path = OUTPUT_DIR / "MORPH_TAGGER_REPORT.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("# Morphological Tagger: Multi-Task Model\n\n")
    f.write("## 1. Source\n")
    f.write(f"- Model: `{model_used}` fine-tuned with four task heads.\n")
    f.write("- 128-dim embedding via learned projection.\n\n")

    f.write("## 2. Training Data\n")
    f.write(f"- **UD Hindi HDTB:** {len(ud_train)} train sentences (after merge).\n")
    if ldc_data:
        f.write(f"- **LDC2010T24:** {len(ldc_data)} sentences merged.\n")
    f.write(f"- **Total supervised:** {len(ud_train)} train / {len(ud_dev)} dev.\n\n")

    f.write("## 3. Inference-Time Procedure\n")
    f.write("At each decoding step, the tagger receives the caption prefix, predicts "
            "POS/Gender/Number/Case for the next token, and produces a 128-dim embedding "
            "`m_i` fed to MAA. No gold tags are required at test time.\n\n")

    f.write("## 4. Independent Evaluation (UD Hindi test split)\n")
    for task, report in reports.items():
        f.write(f"### {task}\n```\n{report}\n```\n\n")

    f.write("## 5. Semi-Automatic Annotation (IndicCorpus)\n")
    if indic_sentences:
        f.write(f"- Pre-annotated {len(indic_sentences)} sentences from AI4Bharat IndicCorpus.\n")
        f.write(f"- Saved to `indiccorpus_pre_annotated.jsonl`.\n")
        f.write("- Manual correction of a subset can expand training data.\n\n")
    else:
        f.write("- IndicCorpus not loaded (network or path issue).\n\n")

    f.write("## 6. Inter-Annotator Agreement\n")
    f.write(f"- Fleiss' Kappa (placeholder): {kappa:.4f}\n")
    f.write("- Replace with real three-annotator ratings.\n")

print(f"\n✅ Report written to: {report_path}")

✅ Dependencies ready.

[1/7] Loading UD Hindi HDTB ...
   ⚠ datasets library failed; using GitHub zip ...
   ✓ UD HDTB: train=13306, dev=1659, test=1684

[2/7] Loading LDC2010T24 ...
   ⚠ LDC2010T24 not found on disk.
   → LDC2010T24 requires a license from the Linguistic Data Consortium.
   → If you have the files, place them in:
      D:\DATASET On 10-12-2025\Objective 1\OB1.1\Dataset\Morph_Tagger_Report\ldc2010t24
   → Continuing with UD HDTB only for supervised training.

[3/7] Loading AI4Bharat IndicCorpus (for semi-automatic annotation) ...
   This corpus is unannotated. It will be used to pre-annotate
   Hindi sentences with the trained tagger, producing a larger
   pool that can be manually corrected for future iterations.
   Attempt 1/3 ...
   ⚠ HTTP Error 403: Forbidden
   Attempt 2/3 ...
   ⚠ HTTP Error 403: Forbidden
   Attempt 3/3 ...
   ⚠ HTTP Error 403: Forbidden
   ⚠ Could not download IndicCorpus. Skipping semi-automatic step.

[4/7] Building label sets ...
   POS: 16 